# Calling the DRS API using Curl

## Set Up Environment
This will be a simple script that uses `curl` to do a `GET` request to the DRS API

You will need to set a few variables:
- DRS_API_KEY
- doc_type
- USER_AGENT

In [ ]:
from dotenv import load_dotenv
import os
import curl

load_dotenv()

#Specified in .env file
DRS_API_KEY = os.getenv("DRS_API_KEY")
BASE_DRS_URL = os.getenv("BASE_DRS_URL")
GENERIC_USER_AGENT = os.getenv("GENERIC_USER_AGENT")
MAX_REQUEST_TIME = os.getenv("MAX_REQUEST_TIME") #fractional seconds

"""
Possible doc_type values (k,v):
    ADFRAWD: ADs (Airworthiness Directives)
    AC: ACs (Advisory Circulars)
    .
    .
    .
    For more see the FAA Master Output
"""

doc_type = "ADFRAWD"

url = f"{BASE_DRS_URL}{doc_type}"

# print(f"URL: {url}\nAPI Key: {DRS_API_KEY}\nUser-Agent: {GENERIC_USER_AGENT}\nDoc Type: {doc_type}\nMax Time: {MAX_REQUEST_TIME}")

For this curl command we have set a number of flags:
 - -v enables "verbosity" to get more information during the curl request to help debugging
 - -H (or --header) to send custom headers
   - Notes: 
        - Use `--header "x-api-key: xxx"` as shown in the documentation
        - Use `-H "USER_AGENT: {user_agent}"` This **is not** in the documentation
            - **Does not work:**`--user-agent {user_agent}`
            - **Does not work:**`-H User-Agent: {user_agent}`

## Using Curl (why??)

<!-- !curl -v \ #verbose
    --location \ #Follows any redirects
    -H f"x-api-key: {DRS_API_KEY}" \ # Sets custom x-api-header
    -H f"USER_AGENT: {GENERIC_USER_AGENT}" \ # Sets custom USER_AGENT header
    --max-time 30 \ # Exits if there is no response after 30 seconds
    --url f"{url}" -->

### Use curl to call the API and get the data back in memory

In [ ]:
import subprocess

command=f"""curl \
--location \
-H \"x-api-key: {DRS_API_KEY}\" \
-H \"USER_AGENT: {GENERIC_USER_AGENT}\" \
--max-time {MAX_REQUEST_TIME} \
--url {url}
"""

response = subprocess.check_output(command, shell=True, text=True)


### Serialize Data & Inspect the Response

In [ ]:
import json
data = json.loads(response)

keys=data.keys()


print("Let's look at our response so we're familiar.")
for k,v in data.items():
    print(f"{k}: {type(v)}")


#### Summary

In [ ]:
summary = data['summary']
documents = data['documents']

for k, v in summary.items():
    print(f"{k}: {v}")

#### Documents

We know that we should have 750 documents so we won't print out everything. 

We also know that documents is actually a list so let's take a look at the first one.

Note: We will truncate the output to the first 100 characters

In [ ]:
documents = data['documents']

doc_1 = documents[0]
none_columns=[]
def truncate_response(k,v):
    if v is not None:
        print(f"{k}: {v[0:100]}")
    else:
        print(f"{k}")
        none_columns.append(k)

for k, v in doc_1.items():
    truncate_response(k,v)
    
print(f"\n\nThe following {len(none_columns)} columns had a value of None: \n\t{none_columns}")

#### Handling Errors

We have to deal with the values the are 'None' but this is good. We might realize that some or several of the columns are unnecessary

In [ ]:
remove_list=[]
for k, v in doc_1.items():
    if v is None:
        remove_list.append(k)
    else:
        print(f"{k}: {v[0:100]}")   
        
print(f"\n\nThe following keys were associated with a NoneType value: {', '.join(remove_list)}")

### What can the Summary tell us?

In [ ]:
summary = data["summary"]
documents = data["documents"]

print(f"Length of Summary: {len(summary)}\nKeys: {summary.keys()}")

print(f"\n{'*'*10} Summary {'*'*10}")
for k, v in summary.items():
    print(f"{k}: {v}")
    

total_docs=summary['totalItems']
count = summary['count']

num_requests = total_docs%count

(quotient, modulus) = (total_docs // count, total_docs%count)

print(f"{'*'*60}\nTotal Number of Docs: {total_docs}\nDocs Received per Request: {count}\n Requests Remaining: {quotient+1}")

print(f"\n\nWe now know how many requests we have remaining and total requests needed. We can now loop through and increase offset by 750 each time to collect all ADs")

In [ ]:
import math
requests_remaining=math.ceil(total_docs / count)

## Get All ADs

The command for getting the rest of the ADs will be very similar to the previous command but we will be adding an offset.

There two obvious ways to do this: 

1. Iterate from 1:26 
2. Check the last response and if 'hasMoreItems" is still true continue

### Calculate Iterations

In [ ]:
import math
requests_remaining_offset=math.floor(total_docs / count)

def construct_command(offset, base_url):
    url = "{}&offset={}".format(base_url, offset)
    command = f"""\
curl \
--location \
-H "x-api-key: {DRS_API_KEY}" \
-H "USER_AGENT: {GENERIC_USER_AGENT}" \
--max-time {MAX_REQUEST_TIME} \
--url {url}\
    """
    return command
    
iteration=0
offset=750
increment=750
total_iterations = math.floor(total_docs/count)
base_url = url

for _ in range(total_iterations):
    iteration += 1
    # print(f"{'*'*60}\nIteration:{iteration}")
    command=construct_command(offset, base_url)
#     url_offset = "{}&offset={}".format(base_url, offset)
#     print(url_offset)
    
    offset += increment
    
    # print(f"Command:{command}")
    

In [ ]:
list_docs=[]
list_docs.extend(documents)
print(len(list_docs))

In [ ]:
### Check hasMoreItems
import math
import json
import time

start=time.time()


requests_remaining_offset=math.floor(total_docs / count)

def construct_command(offset, base_url):
    url = "{}?offset={}".format(base_url, offset)
    command = f"""\
curl \
--location \
-H "x-api-key: {DRS_API_KEY}" \
-H "USER_AGENT: {GENERIC_USER_AGENT}" \
--max-time {MAX_REQUEST_TIME} \
--url {url}\
    """
    return command

def is_moreItems(summary):
    if summary['hasMoreItems']:
        return summary['hasMoreItems']
    else:
        return False
    
iteration=0
offset=0
increment=750
base_url = url
has_more_items = True

document_list=[]
document_list.extend(documents)

response_time_list=[]

while has_more_items:
    iteration +=1
    print(f"{'*'*60}\nIteration:{iteration}\n# Docs Collected: {len(document_list)}")
    offset += increment
    command=construct_command(offset, base_url)
    # print(f"Command: {command}")
    
    
    

    try:
        send_request_time = time.time()
        response = subprocess.check_output(command, shell=True, text=True)
        response = json.loads(response)
        
        summary = response['summary']
        documents = response['documents']
        
        
        has_more_items = is_moreItems(summary)
        print(has_more_items)
        
        document_list.extend(response['documents'])
        print(len(document_list))
        
        total_items = summary['totalItems']
        print(total_items)
        
        count = summary['count']
        print(count)
        
        print(f"Total Items: {total_items}\nCount: {count}, Remaining: {total_items-summary['offset']}\nOffset:{summary['offset']}")
        if not has_more_items:
            print("No more documents remaining...")
            has_more_items = False
            break
    except Exception as e:
        print("Failed to process with Exception: {e}")
        has_more_items=False
        break
    current_time = time.time()
    response_time = time.time() - send_request_time
    elapsed_time= current_time - start
    response_time_list.append(response_time)
    print(f"Total Elapsed Time: {elapsed_time}\nRequest & Process Time: {response_time}")

end=time.time()
total_time=end-start

print(f"Total Run time for collection: {total_time}")
    

    


### Error Checking is Important

As we can see from the last call the script failed fairly early. This was because the machine lost connection to the internet so after 60 seconds it exited the process and tried to do json.loads() on an empty response.

But we still got 7500 ADs!

In [ ]:

# len(document_list)
len(document_list)

## Polars
Make sure you have installed `polars`


`!pip install polars` 

OR

`!pip install 'polars[all]'`

In [ ]:
import polars as pl
# Use data['documents'] if you only pulled the first 750
# df = pl.DataFrame(data['documents'])
# Use document_list for the full list of ADs
df = pl.DataFrame(document_list)
df.columns = ["document_number", "status", "docket_number", "amendment", "responsible_office", "make", "model", "product_type", "product_sub_type", "subject", "affected_ads", "citation", "page_number", "title", "action", "summary", "address", "further_info_contact", "supplementary_info", "regulatory_text", "footer", "part_number", "sub_part", "section_number", "citation_publish_date", "issue_date", "comments", "effective_date", "ab_reference", "ad_reference", "car_reference", "ex_reference", "sfar_reference", "last_modified_date", "document_guid", "document_URL"]

In [ ]:
import polars.selectors as cs

df = df.drop([cs.by_dtype(pl.Null), cs.contains(("EMPTY", "null")), cs.by_name(("sub_part", "section_number", "comments"))])

### Save the dataframe

In [ ]:
import os

def get_save_path(file_name):
    
    base_path=os.getcwd()
    
    # file_name = "all_ads.parquet"
    save_path = "{}/assets/{}".format(base_path, file_name)
    return save_path
# print(save_path)

In [ ]:
# Here we will save to a parquet file.

import os
save_path=get_save_path("all_ads.parquet")
print(save_path)

try:
    df.write_parquet("{save_path}")
    print(f"Saved to: {save_path}")
except:
    df.write_parquet("all_ads.parquet")
    print("Saved to current directory as `all_ads.parquet`")

In [ ]:
# import polars.selectors as cs
# df = df.drop([cs.by_dtype(pl.Null), cs.contains(("EMPTY", "null")), cs.by_name(("sub_part", "section_number", "comments"))])

In [ ]:
df_pa22=df.filter(pl.col("model").list.contains("PA-22-135"))
df_pa22.head()

## Sort the dataframe by document number

# Read from Parquet & Filter

## Separate Actions
- Read the parquet file
- filter for PA-22-135

In [ ]:
import os
import polars as pl

path = os.getcwd()

df=pl.read_parquet("/home/stubbs/ampro/ampro-python/misc/all_ads.parquet")

In [ ]:
import time

def filter_in_array(df):
    df_pa22=df.filter(pl.col("model").list.contains("PA-22-135"))
    return df_pa22
start = time.time()

df_pa22 = filter_in_array(df)
end = time.time()

print(df_pa22)
print(f"Time to filter: {end-start}")
# df_pa22

## Run Both

In [ ]:
import os
import polars as pl
import time

start = time.time()

def filter_in_array(df):
    df_pa22=df.filter(pl.col("model").list.contains("PA-22-135"))
    return df_pa22

path = os.getcwd()
df=pl.read_parquet("/home/stubbs/ampro/ampro-python/misc/all_ads.parquet")

df_pa22 = filter_in_array(df)

name="pa-22-135_all_ads.parquet"
df_pa22.write_parquet(name)
end = time.time()

print(f"{'*'*60}\n\nTotal Time to read from file & filter: {end-start} seconds\n\n{'*'*60}")
print(df_pa22)

